# Beam Dataset Generation for Language Model Training

This notebook generates synthetic beam mechanics datasets for training language models on structural engineering problems. It creates beam configurations with varying parameters, solves them symbolically, and generates question-answer pairs using LLMs.

## Overview

The dataset generation process involves:
1. Creating beam configurations with symbolic parameters (lengths, loads, supports)
2. Solving beam equations symbolically to obtain reactions, moments, and deflections
3. Generating natural language questions using LLMs
4. Extracting ground-truth answers from the solved beam equations
5. Formatting everything into a HuggingFace-compatible dataset

## Workflow

1. **Setup**: Import dependencies and configure symbeam path
2. **Configuration**: Set beam parameters (lengths, loads, positions) and output options
3. **Generation**: Create beam configurations and solve symbolically
4. **Conversion**: Transform to HuggingFace Dataset format
5. **QA Generation**: Generate questions using LLM (optional)
6. **Post-processing**: Extract answers and rename columns
7. **Output**: Save intermediary datasets locally, upload final dataset to Hub

## Output Structure

- **Intermediary datasets** are saved to `intermediary_datasets/` subdirectory:
  - `01_initial_dataset.json` - Initial beam analysis dataset with symbolic solutions
  - `02_dataset_with_qa.json` - Dataset with LLM-generated questions
  - `03_final_dataset.json` - Final processed dataset with cleaned queries and answers
- **Final dataset** is uploaded to HuggingFace Hub for public access

## Requirements

- Python 3.8+
- symbeam library (for symbolic beam analysis)
- HuggingFace datasets library
- vLLM (for LLM inference, optional if skipping QA generation)


### Setup

Import dependencies and configure symbeam path.

In [ ]:
# Standard library imports
import sys
import os
import json

# Scientific computing libraries
import numpy as np  # For numerical operations and array handling
import sympy  # For symbolic mathematics
from sympy import Basic  # Base class for SymPy objects

# HuggingFace datasets library for dataset creation and management
from datasets import Dataset


In [ ]:
## 2. Configure symbeam Library Path

# Determine the current directory (works in both script and notebook contexts)
current_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
symbeam_path = os.path.join(current_dir, "symbeam_v2")

# Add symbeam_v2 to Python path if it exists, otherwise raise an error
if os.path.exists(symbeam_path):
    sys.path.insert(0, os.path.abspath(symbeam_path))
else:
    raise FileNotFoundError(f"symbeam_v2 directory not found at {symbeam_path}")

# Import symbeam library for symbolic beam analysis
from symbeam import beam

# Import symbolic variables for beam parameters
# L = length, E = Young's modulus, I = moment of inertia
# P = point load, M = moment, q = distributed load, x = position along beam
from sympy.abc import L, E, I, P, M, q, x

### Configuration

Set beam parameters (lengths, loads, positions) and output options.

In [ ]:
## 3. Configuration: Beam Dataset Generation Parameters

# ============================================================================
# BEAM PARAMETERS
# ============================================================================
# These parameters define the beam configurations to generate.
# The script will create all combinations of lengths × loads × positions.

lengths = ["L", "2*L", "3*L"]
loads = ["-P", "-2*P", "-3*P"]

# Number of load positions along the beam (normalized 0.0 to 1.0)
# Loads will be placed at evenly spaced positions from start to end
n_positions = 21

# Distributed load value (currently not used, set to 0)
q_val = 0

# Point moment value (currently not used, set to 0)
M_val = 0

# Material properties (kept symbolic for general solutions)
E_val = "E"  # Young's modulus
I_val = "I"  # Moment of inertia

# ============================================================================
# OUTPUT CONFIGURATION
# ============================================================================

# Base filename for the final dataset
output_filename = "BeamRL-TrainData.json"

# Directory where intermediary datasets will be saved
# This directory will be created automatically if it doesn't exist
intermediary_dir = "intermediary_datasets"

# ============================================================================
# HUGGINGFACE HUB CONFIGURATION
# ============================================================================

# Repository name on HuggingFace Hub (format: username/dataset-name)
repo_name = f"tphage/{output_filename.replace('.json', '')}"

# Set to True if you want the dataset to be private
private = False

# HuggingFace token (set to None to use cached credentials from huggingface-cli login)
hf_token = None

# ============================================================================
# DISPLAY CONFIGURATION SUMMARY
# ============================================================================

total_configs = len(lengths) * len(loads) * n_positions
print(f"Configuration Summary:")
print(f"  - Total beam configurations: {total_configs}")
print(f"  - Beam lengths: {lengths}")
print(f"  - Point loads: {loads}")
print(f"  - Load positions per configuration: {n_positions}")
print(f"  - Intermediary datasets directory: {intermediary_dir}/")
print(f"  - Final dataset Hub repository: {repo_name}")

### Generation

Create beam configurations and solve symbolically.

In [ ]:
def create_beam_configuration(P_val=-P, q_val=0, M_val=0, L_val=L, E_val=E, I_val=I, L_loc=0.5):
    """
    Create a beam with specified parameters and solve symbolically.
    
    This function creates a simply supported beam (pin at start, roller at end),
    applies loads, and solves for reactions, moments, and deflections.
    
    Args:
        P_val: Point load value (default: -P, negative indicates downward)
        q_val: Distributed load value (default: 0)
        M_val: Point moment value (default: 0)
        L_val: Beam length (default: L, symbolic)
        E_val: Young's modulus (default: E, symbolic)
        I_val: Moment of inertia (default: I, symbolic)
        L_loc: Normalized load position (0.0 to 1.0, default: 0.5)
    
    Returns:
        tuple: (plot_data, subs_dict)
            - plot_data: Dictionary containing solved beam data (reactions, moments, deflections)
            - subs_dict: Dictionary of symbolic substitutions used in solving
    """
    # Validate load position (must be between 0 and 1)
    if L_loc < 0 or L_loc > 1:
        raise ValueError(f"L_loc must be between and including 0 and 1, got {L_loc}")
    
    # Convert normalized position to actual coordinate
    L_loc_val = L_loc * L_val

    # Create a new beam instance with specified length
    new_beam = beam(L_val)
    
    # Set material properties (constant along the beam)
    new_beam.set_young(0, L_val, E_val)  # Young's modulus
    new_beam.set_inertia(0, L_val, I_val)  # Moment of inertia
    
    # Add supports: pin at start (x=0), roller at end (x=L)
    new_beam.add_support(0, 'pin')
    new_beam.add_support(L_val, 'roller')
    
    # Add point load at specified location
    new_beam.add_point_load(L_loc_val, P_val)
    
    # Solve the beam symbolically
    # subs_dict contains the symbolic parameters for substitution
    subs_dict = {'P': P_val, 'q': q_val, 'L': L_val, 'M': M_val, 'E': E_val, 'I': I_val}
    plot_data = new_beam.solve_v3(subs=subs_dict)
    
    return plot_data, subs_dict

def make_json_serializable(obj):
    """
    Recursively convert SymPy objects to JSON-serializable types.
    
    SymPy symbolic expressions cannot be directly serialized to JSON.
    This function converts them to strings or floats where possible.
    
    Args:
        obj: Object that may contain SymPy Basic objects
    
    Returns:
        JSON-serializable version of the object
    """
    # If it's a SymPy Basic object (symbol, expression, etc.)
    if isinstance(obj, Basic):
        # Try to evaluate to a float if it's a numeric expression
        try:
            return float(obj)
        except Exception:
            # Otherwise convert to string representation
            return str(obj)
    # Recursively process dictionaries
    elif isinstance(obj, dict):
        return {make_json_serializable(k): make_json_serializable(v) for k, v in obj.items()}
    # Recursively process lists, tuples, and sets
    elif isinstance(obj, (list, tuple, set)):
        return [make_json_serializable(i) for i in obj]
    # Return primitive types as-is
    else:
        return obj

def make_json_serializable(obj):
    """Recursively convert SymPy objects to JSON-serializable types."""
    if isinstance(obj, Basic):
        # Try to evaluate to float if possible, else use str
        try:
            return float(obj)
        except Exception:
            return str(obj)
    elif isinstance(obj, dict):
        return {make_json_serializable(k): make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple, set)):
        return [make_json_serializable(i) for i in obj]
    else:
        return obj

In [ ]:
def generate_beam_dataset(lengths=None, loads=None, n_positions=21, q_val=0, M_val=0, E_val=E, I_val=I):
    """
    Generate beam configurations for all combinations of parameters.
    
    This function creates all possible combinations of beam lengths, loads, and load positions,
    solves each configuration symbolically, and collects the results.
    
    Args:
        lengths: List of beam lengths (default: [L, 2*L])
        loads: List of point loads (default: [-P, -2*P])
        n_positions: Number of load positions along beam (default: 21)
        q_val: Distributed load value (default: 0)
        M_val: Point moment value (default: 0)
        E_val: Young's modulus (default: E)
        I_val: Moment of inertia (default: I)
    
    Returns:
        dict: Dictionary containing list of beam configurations with their solved data
    """
    # Set default values if not provided
    if lengths is None:
        lengths = [L, 2*L]
    if loads is None:
        loads = [-P, -2*P]

    # Generate evenly spaced load positions from 0.0 to 1.0
    L_loc_values = np.linspace(0, 1, n_positions)
    
    # Calculate total number of configurations to generate
    total_configs = len(lengths) * len(loads) * n_positions
    print(f"Generating {total_configs} beam configurations...")

    # Initialize data structure to store all configurations
    all_beam_data = {'beam_configurations': []}
    config_id = 0
    
    # Generate all combinations: length × load × position
    for L_val in lengths:
        for P_val in loads:
            for L_loc in L_loc_values:
                try:
                    # Progress indicator
                    print(f"Processing config {config_id+1}/{total_configs}: L={L_val}, P={P_val}, Load pos={L_loc:.3f}")
                    
                    # Create and solve beam configuration
                    plot_data, subs_dict = create_beam_configuration(
                        L_val=L_val, P_val=P_val, L_loc=L_loc, 
                        q_val=q_val, M_val=M_val, E_val=E_val, I_val=I_val
                    )
                    
                    # Convert SymPy objects to JSON-serializable format
                    plot_data_serializable = make_json_serializable(plot_data)
                    subs_dict_serializable = make_json_serializable(subs_dict)
                    
                    # Store configuration data
                    config_entry = {
                        'configuration_id': config_id,
                        'load_positions': [L_loc],  # List for potential multiple loads
                        'parameters': subs_dict_serializable,
                        'plot_data': plot_data_serializable
                    }
                    all_beam_data['beam_configurations'].append(config_entry)
                    config_id += 1
                    
                except Exception as e:
                    # Skip configurations that fail (e.g., loads at support positions)
                    print(f"Error processing config {config_id+1}: {e}")
                    continue
    
    print(f"\nSuccessfully processed {len(all_beam_data['beam_configurations'])} configurations")
    return all_beam_data

In [ ]:
def parse_value(val):
    """
    Parse a value that may be a string, number, or SymPy expression.
    
    This function handles the conversion of configuration parameters that may be
    provided as strings (e.g., "2*L") into SymPy expressions.
    
    Args:
        val: Value to parse (int, float, string, or SymPy expression)
    
    Returns:
        Parsed value (number or SymPy expression)
    """
    # If already a number, return as-is
    if isinstance(val, (int, float)):
        return val
    
    # Try to parse as a float
    try:
        return float(val)
    except Exception:
        # Try to parse as a SymPy expression (e.g., "2*L" -> 2*L)
        try:
            return sympy.sympify(val, locals={"L": L, "P": P, "E": E, "I": I, "M": M, "q": q})
        except Exception:
            # Return original value if parsing fails
            return val

def create_huggingface_dataset(
    lengths=None,
    loads=None,
    n_positions=21,
    q_val=0,
    M_val=0,
    E_val="E",
    I_val="I"
):
    """Generate the beam dataset for all symbolic lengths and loads and prepare it for Hugging Face Hub"""
    print("Generating beam dataset...")
    # Parse values
    lengths = [parse_value(l) for l in (lengths or [L, 2*L])]
    loads = [parse_value(p) for p in (loads or [-P, -2*P])]
    q_val = parse_value(q_val)
    M_val = parse_value(M_val)
    E_val = parse_value(E_val)
    I_val = parse_value(I_val)

    # Patch generate_beam_dataset to accept n_positions, q_val, M_val, E_val, I_val
    beam_data = generate_beam_dataset(
        lengths=lengths,
        loads=loads,
        n_positions=n_positions,
        q_val=q_val,
        M_val=M_val,
        E_val=E_val,
        I_val=I_val
    )
    configurations = beam_data['beam_configurations']
    dataset_data = []
    for config in configurations:
        plot_data = config['plot_data']
        row = {
            'configuration_id': config['configuration_id'],
            'load_position': config['load_positions'][0],
            'parameters': json.dumps(config['parameters']),
            'x_coordinates': plot_data['x_coord'],
            'shear_force': plot_data['shear_force'],
            'bending_moment': plot_data['bending_moment'],
            'slope': plot_data['slope'],
            'deflection': plot_data['deflection'],
            'shear_force_info': json.dumps(plot_data['shear_force_info']),
            'bending_moment_info': json.dumps(plot_data['bending_moment_info']),
            'slope_info': json.dumps(plot_data['slope_info']),
            'deflection_info': json.dumps(plot_data['deflection_info']),
            'points': json.dumps(plot_data['points']),
            'segments': json.dumps(plot_data['segments']),
            'reactions': json.dumps(plot_data['reactions']),
            'internal_loads': json.dumps(plot_data['internal_loads']),
            'deflections': json.dumps(plot_data['deflections'])
        }
        dataset_data.append(row)
    dataset = Dataset.from_list(dataset_data)
    dataset.info.description = """
    Beam analysis dataset with varying symbolic lengths (L, 2L) and loads (-P, -2P), each with 20 load positions.
    Each row contains:
    - configuration_id: Unique identifier for the configuration
    - load_position: Normalized position of the point load (0.0 to 1.0)
    - parameters: JSON string of all beam parameters (symbolic)
    - x_coordinates: List of x-coordinates for plotting
    - shear_force: List of shear force values
    - bending_moment: List of bending moment values
    - slope: List of slope/rotation values
    - deflection: List of deflection values
    - *_info: JSON strings containing local extrema and null points for each quantity
    """
    dataset.info.license = "MIT"
    dataset.info.homepage = "https://github.com/your-repo/beam-analysis"
    return dataset

def upload_to_hub(dataset, repo_name, token=None, private=False, output_filename=None):
    """Upload the dataset to Hugging Face Hub and optionally save locally"""
    if output_filename:
        print(f"Saving dataset locally to {output_filename}...")
        dataset.to_json(output_filename)
    print(f"Uploading dataset to {repo_name}...")
    dataset.push_to_hub(
        repo_name,
        token=token,
        private=private
    )
    print(f"Dataset successfully uploaded to https://huggingface.co/datasets/{repo_name}") 

In [ ]:
# ============================================================================
# Generate Initial Beam Analysis Dataset
# ============================================================================
# This creates the initial dataset by generating all beam configurations
# and solving them symbolically. The dataset includes reactions, moments,
# deflections, and other beam analysis data.

# Generate the HuggingFace dataset using configured parameters
dataset = create_huggingface_dataset(
    lengths=lengths,
    loads=loads,
    n_positions=n_positions,
    q_val=q_val,
    M_val=M_val,
    E_val=E_val,
    I_val=I_val
)

# ============================================================================
# Save Intermediary Dataset
# ============================================================================

# Create intermediary directory if it doesn't exist
os.makedirs(intermediary_dir, exist_ok=True)

# Save the initial dataset to the intermediary directory
# This preserves the state before QA generation
intermediary_filename = os.path.join(intermediary_dir, "01_initial_dataset.json")
print(f"\nSaving intermediary dataset to {intermediary_filename}...")
dataset.to_json(intermediary_filename)
print(f"Saved intermediary dataset: {intermediary_filename}")

### LLM-Based Question Generation

This section uses a language model to generate natural language questions from the beam configurations. The LLM takes beam parameters as input and generates questions asking about reaction forces at the supports.

In [ ]:
# Import LLM inference library (vLLM for efficient GPU inference)
from vllm import LLM, SamplingParams

# Import custom prompt generation functions
from beam_prompt_calculator import *

In [ ]:
# ============================================================================
# LLM Configuration for QA Generation
# ============================================================================
# Configure the language model for question generation.
# The model generates multiple question variations per beam configuration.

LLM_CONFIG = {
    "model_name": "RedHatAI/DeepSeek-R1-Distill-Qwen-7B-quantized.w8a8",  # Quantized model for efficiency
    "max_length": 5120,  # Maximum sequence length
    "temperature": 0.6,  # Controls randomness (lower = more deterministic)
    "top_p": 0.9,  # Nucleus sampling parameter
    "n_outputs": 4  # Number of question variations to generate per sample
}

In [ ]:
# ============================================================================
# System Prompt for Question Generation
# ============================================================================
# This prompt instructs the LLM on how to generate questions about beam
# reaction forces. The model uses chain-of-thought reasoning (with redacted
# reasoning tags) to generate high-quality questions.

PROMPT_Q_SYSTEM = """
You are a question generation assistant. You will be given information about the setup of a statically loaded beam. Your task is to generate a question that asks the reader to calculate the reaction forces at the supports.

Generate a single, self-contained question that includes all the provided details from the setup below. All details are correct. The question should be short and concise. Use limited time reasoning.

The question needs to state the length of the beam, the location and type of the supports of the beam, and the location, magnitude, direction and type of the loads applied to the beam.

Following the think tag concluding the reasoning section, return only the question and no other dialogue referencing the prompt or the setup.
"""

In [ ]:
class LLMManager:
    """Manages LLM model loading and generation."""
    
    def __init__(self, config):
        self.config = config
        self.llm = None
        self.sampling_params = None
    
    def load_model(self):
        """Load the LLM model using vLLM."""
        print(f"Loading LLM model: {self.config['model_name']}...")
        self.llm = LLM(
            model=self.config['model_name'],
            max_model_len=self.config['max_length'],
            dtype="half"  # Use float16 for GPU compatibility (works on Quadro RTX 5000 and L4 GPUs)
        )
        self.sampling_params = SamplingParams(
            temperature=self.config['temperature'],
            max_tokens=self.config['max_length'],
            top_p=self.config['top_p'],
            n=self.config['n_outputs']
        )
        print("Model loaded successfully!")
    
    def generate_responses(self, prompts, system_prompt):
        """Generate responses for a batch of prompts."""
        # Add <think> if not already present
        full_prompts = []
        for prompt in prompts:
            if "<think>" not in prompt:
                full_prompt = system_prompt + "\n\n" + prompt + "\n<think>\n"
            else:
                full_prompt = system_prompt + "\n\n" + prompt + "\n"
            full_prompts.append(full_prompt)
        
        outputs = self.llm.generate(full_prompts, self.sampling_params)
        
        results = []
        for output in outputs:
            prompt_responses = []
            for single_output in output.outputs:
                generated_text = single_output.text.strip()
                prompt_responses.append(generated_text if generated_text else "No response generated")
            results.append(prompt_responses)
        
        return results

In [ ]:
def prompt_text_from_load_and_params_custom(sample):
    """
    Generate prompt text using only load_position and parameters columns.
    Custom version that handles the actual dataset format where P is a string.
    
    Args:
        sample: Dictionary containing the sample data with load_position and parameters columns
        
    Returns:
        str: The prompt text combining load position and parameters information
    """
    load_position = sample.get("load_position")
    parameters_str = sample.get("parameters")
    
    # Parse parameters if it's a JSON string
    if isinstance(parameters_str, str):
        try:
            parameters = json.loads(parameters_str)
        except (json.JSONDecodeError, TypeError):
            parameters = {}
    else:
        parameters = parameters_str
    
    # Extract individual parameters
    P = str(parameters.get("P", ""))
    q = str(parameters.get("q", ""))
    L = str(parameters.get("L", ""))
    M = str(parameters.get("M", ""))
    E = str(parameters.get("E", ""))
    I = str(parameters.get("I", ""))

    prompt_parts = []

    prompt_parts.append(f"The beam has a length of {L}.")
    prompt_parts.append(f"The beam has a Young's modulus of {E}.")
    prompt_parts.append(f"The beam has a moment of inertia of {I}.")
    
    # Calculate load location from load_position
    if L != "L":
        L_unitless = float(L.replace("*L", ""))  # Strip "*L" from string and convert to float
        location_of_load = float(load_position) * L_unitless
    else:
        location_of_load = float(load_position)

    # Add point load information
    if P and P != "0":
        prompt_parts.append(f"There is an applied point load of {P} at x={location_of_load}*L.")
        if "-" in P:
            prompt_parts.append(f"A negative load means the load is applied downward.")
    
    # Add moment information if present
    if M and M != "0":
        prompt_parts.append(f"There is an applied moment of {M}.")
    
    # Add distributed load information if present
    if q and q != "0":
        prompt_parts.append(f"There is a distributed load of {q}.")
    
    # Add support information (default: pin at 0, roller at L)
    prompt_parts.append(f"The beam has a pin support at x=0 and a roller support at x={L}.")
    
    return "\n".join(prompt_parts)

def generate_qa_for_dataset(dataset, llm_manager):
    """
    Generate questions for a dataset.
    
    Args:
        dataset: HuggingFace Dataset object
        llm_manager: LLMManager instance (must have model loaded)
    
    Returns:
        Dataset with added 'llm_response_Q' and 'prompt_Q' columns
    """
    # Convert to list for easier processing
    dataset_list = list(dataset)
    
    print(f"Generating QA for {len(dataset_list)} samples...")
    
    # Prepare prompts for question generation (all at once)
    prompts_Q = []
    
    for sample in dataset_list:
        # Question generation prompt (simpler, from load_position and parameters)
        # Use custom function that handles the actual data format
        prompt_Q_input = prompt_text_from_load_and_params_custom(sample)
        prompts_Q.append(prompt_Q_input)
    
    # Generate questions for all samples at once
    print("Generating questions...")
    responses_Q = llm_manager.generate_responses(prompts_Q, PROMPT_Q_SYSTEM)
    
    # Add responses to samples
    processed_data = []
    for j, sample in enumerate(dataset_list):
        sample["prompt_Q"] = prompts_Q[j]
        sample["llm_response_Q"] = responses_Q[j]  # List of n_outputs questions (cleaned)
        processed_data.append(sample)
    
    # Create new dataset with QA data
    enhanced_dataset = Dataset.from_list(processed_data)
    print(f"QA generation completed! Dataset now has {len(enhanced_dataset)} samples.")
    
    return enhanced_dataset

In [ ]:
# Initialize LLM manager
llm_manager = LLMManager(LLM_CONFIG)

# Load the model (this may take a few minutes)
llm_manager.load_model()

# Generate questions and reasoning traces (all samples at once)
dataset_with_qa = generate_qa_for_dataset(
    dataset=dataset,
    llm_manager=llm_manager
)

# Update your dataset variable
dataset = dataset_with_qa

# Save QA-enhanced intermediary dataset
qa_filename = os.path.join(intermediary_dir, "02_dataset_with_qa.json")
print(f"\nSaving QA-enhanced dataset to {qa_filename}...")
dataset.to_json(qa_filename)
print(f"Saved intermediary dataset: {qa_filename}")

### Post-Processing: Extract Answers and Clean Dataset

This section extracts ground-truth answers from the solved beam equations and cleans up the dataset by:
- Extracting questions from LLM responses (removing reasoning traces)
- Extracting reaction force answers from the beam solutions
- Renaming columns for consistency
- Saving the final processed dataset

In [ ]:
# ============================================================================
# Post-Processing Functions
# ============================================================================

def extract_question_from_response(response_text):
    """
    Extract the question from an LLM response that may contain reasoning traces.
    
    LLM responses often include reasoning traces wrapped in tags. This function
    extracts only the final question text after the reasoning section.
    
    Args:
        response_text: String containing LLM response with potential reasoning traces
    
    Returns:
        str: Clean question text, or empty string if not found
    """
    # Look for the closing tag that marks the end of reasoning
    closing_tag = '</think>'
    if closing_tag in response_text:
        # Split on the closing tag and take everything after it
        parts = response_text.split(closing_tag)
        if len(parts) > 1:
            question = parts[-1].strip()
            if question:
                return question
    return ""

def extract_cleaned_responses_Q_from_row(row):
    """
    Extract cleaned questions from a dataset row.
    
    Processes the llm_response_Q column which contains a list of raw LLM responses
    and extracts clean questions from each.
    
    Args:
        row: Dataset row containing 'llm_response_Q' field
    
    Returns:
        list: List of cleaned question strings
    """
    llm_resp_list = row["llm_response_Q"]
    if isinstance(llm_resp_list, list):
        return [extract_question_from_response(resp) for resp in llm_resp_list]
    return []

def extract_cleaned_responses_Q_from_row(row):
    llm_resp_list = row["llm_response_Q"]
    if isinstance(llm_resp_list, list):
        return [extract_question_from_response(resp) for resp in llm_resp_list]
    return []

def extract_preferred_answer(reactions_data):
    """
    Extract the reactions array from the reactions column data and format as simple values.
    Expected format: {"header": "Exterior Reactions", "reactions": [...]}
    Returns: List of strings where values are the "value" field from each reaction
    """
    try:
        if isinstance(reactions_data, str):
            reactions_dict = json.loads(reactions_data)
        else:
            reactions_dict = reactions_data
            
        if isinstance(reactions_dict, dict) and "reactions" in reactions_dict:
            reactions = reactions_dict["reactions"]
            if isinstance(reactions, list):
                # Extract just the "value" field from each reaction and remove asterisks
                values = [reaction.get("value", "").replace("*", "") for reaction in reactions]
                return values
        return None
    except (json.JSONDecodeError, TypeError, KeyError):
        return None

def add_answer_column(example):
    """Add answer column by extracting preferred answers from reactions"""
    answer = extract_preferred_answer(example.get("reactions"))
    example["answer"] = answer if answer else []
    return example

In [ ]:
# ============================================================================
# Clean Questions and Extract Answers
# ============================================================================

# Extract cleaned questions from LLM responses (remove reasoning traces)
print("Extracting cleaned questions from LLM responses...")
cleaned_responses_Q = [extract_cleaned_responses_Q_from_row(row) for row in dataset]

# Add cleaned questions as 'query' column and remove intermediate columns
dataset = dataset.add_column("query", cleaned_responses_Q)
dataset = dataset.remove_columns(["llm_response_Q", "prompt_Q"])
print("Renamed column 'llm_response_Q' to 'query' and removed 'prompt_Q'.")

# Extract ground-truth answers from the reactions field
print("\nExtracting answers from reactions column...")
dataset = dataset.map(add_answer_column)
print(f"Added 'answer' column. Sample: {dataset[0]['answer']}")

# ============================================================================
# Save Final Processed Dataset
# ============================================================================

# Save the final dataset locally before uploading to Hub
final_filename = os.path.join(intermediary_dir, "03_final_dataset.json")
print(f"\nSaving final dataset to {final_filename}...")
dataset.to_json(final_filename)
print(f"Saved final dataset: {final_filename}")


In [ ]:
# ============================================================================
# Upload Final Dataset to HuggingFace Hub
# ============================================================================
# Upload the final processed dataset to HuggingFace Hub for public access.
# The dataset includes beam configurations, symbolic solutions, LLM-generated
# questions, and ground-truth answers.

print(f"\nUploading final dataset to HuggingFace Hub: {repo_name}...")
dataset.push_to_hub(repo_name, token=hf_token, private=private)
print(f"Dataset successfully uploaded to https://huggingface.co/datasets/{repo_name}")


In [ ]:
first_row = dataset[0]
last_row = dataset[len(dataset)-1]
all_columns = list(dataset.features)
last_two_cols = all_columns[-2:]

print("First row (last two columns):")
for col in last_two_cols:
    print(f"{col}:")
    for elem in first_row[col]:
        print(elem)
print("Last row (last two columns):")
for col in last_two_cols:
    print(f"{col}:")
    for elem in last_row[col]:
        print(elem)